# Required items
-   School AGEID 
-   Location AGEID
-   Main Location
-   Funded Student FTE
-   FFPOS FTE
-   Latitude/Longitude
-   Statistical Area 1 2021



# School and student FTE extracted from EFS_GOLD. 
Will require 2021 Statistical Area 1 to be obtained separately from the DAP environment as it is not currently available directly.

In [0]:
locations=sql(f"""with 
students as 
(select 
location_ageid,
sum((overseas_student='No')::int*fte::decimal(3,1)*headcount)::decimal(10,1) as funded_student_fte,
sum((overseas_student='Yes')::int*fte::decimal(3,1)*headcount)::decimal(10,1) as ffpos_fte
 from efs_gold.ngc_student_cohort
 group by location_ageid),
 schools as 
 (select distinct ageid,School_Location_AGEID,case when IsPrimaryLocation then 'Yes' else 'No' end as MainLocation,
    Latitude,Longitude
  from efs_gold.schools
  where coalesce(school_location_EffectiveToDate,getdate())>=getdate()) /* Obtain most current school-location connection */
select 
schl.ageid as `School AGEID`,schl.school_location_ageid as `Location AGEID`,schl.mainlocation as `Main Location`,
fte.funded_student_fte as `Funded Student FTE`,fte.ffpos_fte as `FFPOS FTE`,
schl.latitude,schl.longitude
from 
students fte 
left join 
schools schl
on fte.location_ageid=schl.school_location_ageid
 order by schl.ageid,schl.School_Location_AGEID""")

locations.createOrReplaceTempView('locations')

 


# Import locations by SA1 to be updated


In [0]:
import os

# Absolute path of file
sa1_path=os.path.join(os.getcwd(),"Locations by SA1 2021.csv") # ACARA Australian Schools List 

locations_sa1 = spark.read.csv(
f"file:{sa1_path}",
header=True,
inferSchema=True
)

locations_sa1.select(['*']).createOrReplaceTempView('locations_sa1')


In [0]:
RQ6609=sql(f"""
select l.*,sa1.`SA12021.SA1_CODE21` as `Statistical Area 2021`
from locations l
left join locations_sa1 sa1 
on l.`Location AGEID`=sa1.LocationAGEID""")

RQ6609.createOrReplaceTempView('rq6609')

In [0]:
RQ6609.where("`statistical area 2021` is null").display()


School AGEID,Location AGEID,Main Location,Funded Student FTE,FFPOS FTE,latitude,longitude,Statistical Area 2021


In [0]:
RQ6609.display()

School AGEID,Location AGEID,Main Location,Funded Student FTE,FFPOS FTE,latitude,longitude,Statistical Area 2021
17215,57215,Yes,174.0,0.0,-35.249976,149.162530,80105105602
15939,77303,No,186.0,0.0,-42.774680,147.252958,60103101305
1009,41009,Yes,219.0,0.0,-37.685532,144.923358,21005124212
29085,89919,No,25.0,0.0,-34.430489,150.888463,10704154836
512,40512,Yes,312.0,0.0,-27.516591,153.030837,30302105208
856,40856,Yes,508.0,11.0,-37.989561,145.064556,20803119029
17689,57689,Yes,141.0,0.0,-25.311175,152.797454,31904151603
84,40084,Yes,1614.0,9.0,-31.913402,115.775922,50502109621
80108,88477,No,189.0,0.0,-37.807818,144.968346,20604150302
1657,41657,Yes,174.0,0.0,-33.948161,151.121707,11904166907


In [0]:
%sql
with addresses as 
(select SchoolId,SchoolLocationId,bem_ageid.BusinessKeyValue as LocationAGEID,l.AddressId,
o.AddressLine1,o.AddressLine2,o.Suburb,o.Postcode,o.AustralianStateId,o.Latitude,o.Longitude
from efs_silver.org_school_location l 
join efs_silver.org_business_key_entity_map as bem_ageid  /* Identify location AGEID */
                     on bem_ageid.EfsColumnValue=l.schoollocationid and bem_ageid.EfsColumnName='SchoolLocationId'
join efs_silver.org_address o on 
o.AddressId=l.AddressId)
select 
addr.* 
from addresses addr 
join RQ6609 rq 
on rq.`location ageid`=addr.locationageid
and rq.`statistical area 2021` is null

SchoolId,SchoolLocationId,LocationAGEID,AddressId,AddressLine1,AddressLine2,Suburb,Postcode,AustralianStateId,Latitude,Longitude
